In [1]:
import asyncio
from random import random

In [2]:
async def echo(index: int):
    await asyncio.sleep(0.1)
    return index


async def echo_random_latency(index: int):
    await asyncio.sleep(random())
    return index

# Test Executor 

In [3]:
from ragas.executor import is_event_loop_running, as_completed

/Users/mithras/_code/ragas/ragas/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
assert is_event_loop_running() is True, "is_event_loop_running() returned False"

In [5]:
async def _run():
    results = []
    for t in as_completed([echo(1), echo(2), echo(3)], 3):
        r = await t
        results.append(r)
    return results


results = await _run()

expected = [1, 2, 3]
assert results == expected, f"got: {results}, expected: {expected}"

## Test Executor

In [6]:
from ragas.executor import Executor

In [7]:
# test order of results when they should return in submission order
executor = Executor(raise_exceptions=True)
for i in range(10):
    executor.submit(echo, i, name=f"echo_{i}")

results = executor.results()
assert results == list(range(10))

Evaluating: 100%|██████████| 10/10 [00:00<00:00, 97.46it/s]


In [8]:
# test order of results when may return unordered
executor = Executor(batch_size=None)

# add jobs to the executor
for i in range(10):
    executor.submit(echo_random_latency, i, name=f"echo_order_{i}")

# Act
results = executor.results()
# Assert
assert results == list(range(10))

Evaluating: 100%|██████████| 10/10 [00:00<00:00, 11.10it/s]


In [9]:
# Test output order; batching
executor = Executor(batch_size=3)

# add jobs to the executor
for i in range(10):
    executor.submit(echo_random_latency, i, name=f"echo_order_{i}")

# Act
results = executor.results()
# Assert
assert results == list(range(10))

Evaluating: 100%|██████████| 10/10 [00:02<00:00,  3.68it/s]


In [10]:
# Test no progress
executor = Executor(show_progress=False)

# add jobs to the executor
for i in range(10):
    executor.submit(echo_random_latency, i, name=f"echo_order_{i}")

# Act
results = executor.results()
# Assert
assert results == list(range(10))

In [11]:
# Test multiple submission sets
executor = Executor(raise_exceptions=True)
for i in range(1000):
    executor.submit(asyncio.sleep, 0.01)

results = executor.results()
assert results, "Results should be list of None"

for i in range(1000):
    executor.submit(asyncio.sleep, 0.01)

results = executor.results()
assert results, "Results should be list of None"

Evaluating: 100%|██████████| 1000/1000 [00:00<00:00, 1380.20it/s]


# Test Metric

In [12]:
from ragas.metrics.base import Metric


class FakeMetric(Metric):
    name = "fake_metric"
    _required_columns = ("user_input", "response")

    def init(self):
        pass

    async def _ascore(self, row, callbacks) -> float:
        return 0


fm = FakeMetric()

In [13]:
score = fm.score({"user_input": "a", "response": "b"})
assert score == 0

/var/folders/c9/8kpl7v1j2zs5y5cwry4bwpn80000gn/T/ipykernel_2623/3943354553.py:1: DeprecationWarning: The function score was deprecated in 0.2, and will be removed in the 0.3 release. Use single_turn_ascore instead.
  score = fm.score({"user_input": "a", "response": "b"})


# Test run_async_tasks

In [14]:
from ragas.async_utils import run_async_tasks

In [15]:
# run tasks unbatched
tasks = [echo_random_latency(i) for i in range(10)]
results = run_async_tasks(tasks, batch_size=None, show_progress=True)
# Assert
assert sorted(results) == list(range(10))

Running async tasks: 100%|██████████| 10/10 [00:01<00:00, 10.00it/s]


In [16]:
# run tasks batched
tasks = [echo_random_latency(i) for i in range(10)]
results = run_async_tasks(tasks, batch_size=3, show_progress=True)
# Assert
assert sorted(results) == list(range(10))

Running async tasks: 100%|██████████| 10/10 [00:03<00:00,  3.10it/s]


In [17]:
# Test no progress
tasks = [echo_random_latency(i) for i in range(10)]
results = run_async_tasks(tasks, batch_size=3, show_progress=False)
# Assert
assert sorted(results) == list(range(10))